In [ ]:
import json
import sys

import pandas as pd
from pathlib import Path

# Path setup to import from the project root
project_root = Path().resolve().parent.parent.parent
sys.path.insert(0, str(project_root))

from event_synchronization.with_dynamic_events.statsbomb import StatsbombSyncDynamicEventsManager
from event_synchronization.events_utils.statsbomb import StatsbombEvents
from skillcorner.client import SkillcornerClient

skc_client = SkillcornerClient(username='USERNAME', password='PASSWORD')

# Load data

In [ ]:
data_dir = 'path/to/your/skc_dynamic_event/and/sb/match/folder/'

In [ ]:
# load match data with SKC match_id
MATCH_ID = 0
match_data = skc_client.get_match(MATCH_ID)

# load skc dynamic events
skc_events = skc_client.get_dynamic_events(
                match_id=MATCH_ID, params={'file_format': 'csv', 'ignore_dynamic_events_check': False}
            )

# load raw_statsbomb_events
raw_statsbomb_events_path = data_dir + 'sb_events.json'
with open(raw_statsbomb_events_path) as f:
    raw_statsbomb_events = json.load(f)

# load statsbomb_match_data
statsbomb_lineup_path = data_dir + 'sb_lineup.json'
with open(statsbomb_lineup_path) as f:
    statsbomb_lineup = json.load(f)

# Run DynamicEvent Synchronization Process

In [ ]:
statsbomb_home_team_id = None  # or set to an integer team ID if needed
# standardize events
if statsbomb_home_team_id is None:  # new version without home team id
    statsbomb_events = StatsbombEvents(raw_statsbomb_events, statsbomb_lineup, match_data)
else:
    statsbomb_events = StatsbombEvents(raw_statsbomb_events, statsbomb_lineup, match_data, statsbomb_home_team_id)

# apply the whole dynamic events synchronization process
skc_events_mapping, statsbomb_events_mapping = StatsbombSyncDynamicEventsManager(
    skc_events, raw_statsbomb_events, statsbomb_events
).run()